# RunwayGuard RT-DETR training
Run cells in order. The notebook downloads the official FOD-A archive, uses the already-audited grouped split manifests, converts Pascal VOC annotations, performs a 2% smoke test, then launches the full run.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics==8.4.147 gdown==5.2.0 pillow==11.3.0

Upload `RunwayGuard_project.zip`, created from the local `RunwayGuard` folder.

In [ ]:
from google.colab import files
uploaded = files.upload()
project_zip = next(name for name in uploaded if name.lower().endswith('.zip'))
!unzip -q -o "{project_zip}" -d /content
!ls /content/RunwayGuard

In [ ]:
import gdown, pathlib
archive = '/content/foda_v2_1.zip'
gdown.download(id='1RdErcq8PGRXZUOGauaACkQG44T-QyZ4x', output=archive, quiet=False)
!unzip -q -o {archive} -d /content/foda
matches = list(pathlib.Path('/content/foda').rglob('VOC2007'))
assert len(matches) == 1, matches
dataset_root = matches[0]
print(dataset_root)

In [ ]:
import subprocess
subprocess.run([
    'python', '/content/RunwayGuard/scripts/convert_voc_to_yolo.py',
    '--dataset-root', str(dataset_root),
    '--splits-dir', '/content/RunwayGuard/data_splits',
    '--taxonomy', '/content/RunwayGuard/config/taxonomy.json',
    '--output-dir', '/content/foda_yolo',
], check=True)
!cat /content/foda_yolo/data.yaml

## Smoke test
This runs one epoch on 2% of the training set. Continue only if it creates a checkpoint without data, CUDA, or NaN errors.

In [ ]:
!cd /content/RunwayGuard && python train.py --data /content/foda_yolo/data.yaml --epochs 1 --imgsz 480 --batch 4 --device 0 --smoke-test

## Full baseline
Mount Drive so a Colab disconnect does not erase the checkpoint. Start with batch 8 on a T4. If CUDA reports out-of-memory, rerun with batch 4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cd /content/RunwayGuard && python train.py --data /content/foda_yolo/data.yaml --model rtdetr-l.pt --epochs 25 --imgsz 480 --batch 8 --device 0 --seed 42 --project /content/drive/MyDrive/RunwayGuardRuns --name rtdetr_l_foda

## Untouched grouped-test evaluation
Run only after the model and thresholds are fixed.

In [ ]:
best = '/content/drive/MyDrive/RunwayGuardRuns/rtdetr_l_foda/weights/best.pt'
!cd /content/RunwayGuard && python evaluate.py --weights {best} --data /content/foda_yolo/data.yaml --imgsz 480 --batch 8 --device 0 --output /content/drive/MyDrive/RunwayGuardRuns/rtdetr_l_foda/test_metrics.json
!cat /content/drive/MyDrive/RunwayGuardRuns/rtdetr_l_foda/test_metrics.json